# Oral Disease Image Classifier v2 (IT2226)

Classifies an intra-oral photo into one of six classes: **Calculus, Hypodontia, Mouth ulcer, Normal, caries, tooth discoloration**.

This notebook is an upgrade of `AIDL_proj.ipynb`. It is built to run top to bottom on **Google Colab with a T4 GPU** (`Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`).

| Problem in v1 | What v2 does |
|---|---|
| `val_dataset.dataset.transform = ...` also changed the training transform, so augmentation never ran | Separate `Dataset` objects for train and eval, each with its own transform |
| 553 redundant byte-identical copies; 27 pictures (121 files) filed under more than one class | Exact duplicates removed; pictures with conflicting labels dropped |
| Pre-augmented and near-duplicate copies could land in both train and validation | Copies found with image embeddings plus a keypoint (ORB + RANSAC) check; split by group so a picture and its copies stay together |
| Train/validation only, no seed, model saved at the last epoch | Seeded, stratified, group-aware **train / validation / test** split; best epoch by validation macro-F1 is saved; test set used once |
| Accuracy only | Per-class precision, recall, F1, confusion matrix, balanced accuracy, macro ROC-AUC |
| One configuration, frozen backbone | Experiment table comparing a v1-style baseline with fine-tuned MobileNetV3, EfficientNet-B0 and ConvNeXt-Tiny |
| Normal photos are high-resolution phone shots, other classes are small web images (possible shortcut) | Every image is shrunk to the same maximum size before training and inference; results are broken down by source resolution |
| Any upload gets a confident diagnosis; Grad-CAM labelled "Located issue" | Confidence threshold chosen on validation data with an "uncertain" fallback; Grad-CAM labelled as regions that influenced the prediction |

**Not a medical device.** Predictions are for a student project and must not replace a dentist's assessment.

## 1. Setup

In [ ]:
# Colab already ships torch, torchvision, scikit-learn, OpenCV, pandas and matplotlib.
# Only Grad-CAM needs installing.
%pip install -q grad-cam

In [ ]:
import os, sys, json, time, random, hashlib, shutil, copy, subprocess
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, f1_score, roc_auc_score)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ------------------------------------------------------------------
# Configuration. Change values here, not further down.
# ------------------------------------------------------------------
CFG = dict(
    seed=42,
    img_size=224,             # model input size
    max_side=612,             # every image is shrunk so its longest side is <= 612 px (see EDA, section 3)
    batch_size=32,
    num_workers=0 if sys.platform == "win32" else 2,   # Colab gives 2 CPU cores; Windows notebooks need 0
    test_frac=0.15,
    val_frac=0.15,
    near_dup_threshold=0.86,  # embedding similarity at or above which two photos are linked directly (section 4.2)
    geo_min_sim=0.60,         # pairs between this and near_dup_threshold get the keypoint check (section 4.2)
    geo_neighbours=10,        # each photo is checked against its 10 most similar photos
    orb_features=500,         # ORB keypoints per image for the keypoint check
    orb_min_inliers=15,       # keypoint matches that must agree before two photos count as the same
    max_eval_group=30,        # groups with more photos than this always go to training (section 5)
    use_preaugmented=False,   # keep the 875 'Tooth_Discoloration_0_*' pre-augmented files? (section 4.3)
    max_epochs=25,
    patience=6,               # early stopping: epochs without validation macro-F1 improvement
    target_selective_acc=0.95 # accuracy the confidence threshold should reach on validation data (section 8)
)

# SMOKE_TEST runs the whole notebook on a tiny subset in a few minutes to check nothing is broken.
# Leave it False for real training.
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"
if SMOKE_TEST:
    CFG.update(max_epochs=2, patience=2, batch_size=16)

WORK_DIR = Path("/content") if IN_COLAB else Path.cwd()
ART_DIR = WORK_DIR / "artifacts"          # models, figures, split files
CKPT_DIR = ART_DIR / "checkpoints"
FIG_DIR = ART_DIR / "figures"
for d in (ART_DIR, CKPT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"            # mixed precision on the T4 roughly doubles speed
torch.backends.cudnn.benchmark = True

print("Colab:", IN_COLAB, "| SMOKE_TEST:", SMOKE_TEST)
print("torch", torch.__version__, "| device:", device,
      "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "no GPU (training will be slow)")

## 2. Get the dataset

Source: [Oral Image Dataset on Kaggle](https://www.kaggle.com/datasets/jhirshaikh/oral-image-dataset) (licence CC0-1.0).

The cell looks for an existing copy first (`/content/dataset`, or `./datasets` when run locally). Otherwise it downloads from Kaggle. It reads your Kaggle API token from the Colab secret `KAGGLE_API_TOKEN` if you created one (key icon in the left sidebar), or asks you to paste it. The token is never written into the notebook.

In [ ]:
CLASS_DIRS = {"Calculus", "Hypodontia", "Mouth ulcer", "Normal", "caries", "tooth discoloration"}


def find_data_root(base):
    base = Path(base)
    if not base.exists():
        return None
    for root, dirs, _ in os.walk(base):
        if CLASS_DIRS.issubset(dirs):
            return Path(root)
    return None


DATA_ROOT = None
for candidate in [WORK_DIR / "dataset", WORK_DIR / "datasets", Path.cwd() / "datasets", Path.cwd().parent / "datasets"]:
    DATA_ROOT = find_data_root(candidate)
    if DATA_ROOT:
        break

if DATA_ROOT is None:
    if not IN_COLAB:
        raise FileNotFoundError("Dataset not found. Put the 'oral dataset' folder under ./datasets or run on Colab.")
    import getpass
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("KAGGLE_API_TOKEN")
    except Exception:
        pass
    if not token and not Path("~/.kaggle/kaggle.json").expanduser().exists():
        token = getpass.getpass("Paste your Kaggle API token: ")
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token.strip()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"], check=True)
    subprocess.run(["kaggle", "datasets", "download", "-d", "jhirshaikh/oral-image-dataset", "-p", "/content"], check=True)
    subprocess.run(["unzip", "-q", "-o", "/content/oral-image-dataset.zip", "-d", "/content/dataset"], check=True)
    DATA_ROOT = find_data_root(WORK_DIR / "dataset")

assert DATA_ROOT is not None, "Download finished but the class folders were not found."
print("Data root:", DATA_ROOT)
print(sorted(p.name for p in DATA_ROOT.iterdir() if p.is_dir()))

## 3. Exploratory data analysis

Build one table with every file, its class, size, EXIF-corrected dimensions and an MD5 hash of its bytes. The hash finds exact duplicates in section 4.

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
EXIF_ORIENTATION = 0x0112

records, unreadable = [], []
for cls_dir in sorted(p for p in DATA_ROOT.iterdir() if p.is_dir()):
    for f in sorted(cls_dir.iterdir()):
        if f.suffix.lower() not in IMG_EXT:
            continue
        raw = f.read_bytes()
        try:
            with Image.open(f) as im:
                w, h = im.size
                if im.getexif().get(EXIF_ORIENTATION, 1) in (5, 6, 7, 8):   # rotated 90 degrees
                    w, h = h, w
        except Exception as e:
            unreadable.append((str(f), repr(e)))
            continue
        records.append(dict(path=str(f), label=cls_dir.name, file=f.name, width=w, height=h,
                            long_side=max(w, h), kb=len(raw) / 1024, md5=hashlib.md5(raw).hexdigest(),
                            is_preaug=f.name.startswith("Tooth_Discoloration_0_")))

df = pd.DataFrame(records)
CLASSES = sorted(df.label.unique())          # same order torchvision ImageFolder used in v1
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
df["label_idx"] = df.label.map(CLASS_TO_IDX)

print(f"{len(df)} readable images, {len(unreadable)} unreadable")
summary = df.groupby("label").agg(images=("file", "size"),
                                  median_width=("width", "median"), median_height=("height", "median"),
                                  share_long_side_over_1000=("long_side", lambda s: round((s > 1000).mean(), 2)),
                                  pre_augmented=("is_preaug", "sum"))
summary

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
counts = df.label.value_counts().reindex(CLASSES)
ax[0].bar(CLASSES, counts.values, color="tab:blue")
for i, v in enumerate(counts.values):
    ax[0].text(i, v, str(v), ha="center", va="bottom")
ax[0].set_title("Images per class (raw)")
ax[0].tick_params(axis="x", rotation=30)

ax[1].boxplot([df.loc[df.label == c, "long_side"] for c in CLASSES], tick_labels=CLASSES)
ax[1].set_yscale("log")
ax[1].axhline(CFG["max_side"], color="red", ls="--", label=f"max_side = {CFG['max_side']}")
ax[1].set_title("Longest image side in pixels (log scale)")
ax[1].tick_params(axis="x", rotation=30)
ax[1].legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "eda_counts_sizes.png", dpi=120)
plt.show()

**What to look for.**

- **Imbalance.** Calculus and Hypodontia have about 1,250 images each, while caries, Mouth ulcer and Normal have 200 to 300. Accuracy alone would hide poor recall on the small classes, so the notebook reports macro-F1 and per-class recall and uses a class-weighted loss.
- **Resolution shortcut.** Most Normal images are phone photos (several thousand pixels wide). Most other classes are small web images. A model could learn "big sharp photo = Normal". To reduce this, every image is shrunk so its longest side is at most `max_side` before training and at inference time, and section 7.3 breaks test results down by source resolution.

In [ ]:
def show_grid(frame, n_per_class=5, title=None, size=2.6):
    fig, axes = plt.subplots(len(CLASSES), n_per_class, figsize=(n_per_class * size, len(CLASSES) * size))
    for r, c in enumerate(CLASSES):
        sub = frame[frame.label == c]
        sub = sub.sample(min(n_per_class, len(sub)), random_state=CFG["seed"]) if len(sub) else sub
        for k in range(n_per_class):
            a = axes[r, k]
            a.axis("off")
            if k < len(sub):
                with Image.open(sub.iloc[k].path) as im:
                    a.imshow(ImageOps.exif_transpose(im).convert("RGB"))
                if k == 0:
                    a.set_title(c, loc="left", fontsize=11, fontweight="bold")
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()


show_grid(df, title="Random samples per class")

## 4. Data cleaning

### 4.1 Exact duplicates

Files with the same MD5 hash are byte-identical. If all copies have the same label, keep one. If copies are filed under different classes, the label is contradictory and all copies are dropped.

In [ ]:
labels_per_hash = df.groupby("md5").label.nunique()
conflict_hashes = set(labels_per_hash[labels_per_hash > 1].index)
conflicts = df[df.md5.isin(conflict_hashes)].sort_values("md5")

clean = (df[~df.md5.isin(conflict_hashes)]
         .drop_duplicates("md5", keep="first")
         .reset_index(drop=True))

print(f"Raw images:                     {len(df)}")
print(f"Redundant exact copies removed: {len(df) - len(clean) - len(conflicts)}")
print(f"Conflicting-label images dropped: {len(conflicts)} ({len(conflict_hashes)} distinct pictures)")
print(f"Remaining:                      {len(clean)}")
pd.DataFrame({"raw": df.label.value_counts(), "after_exact_dedup": clean.label.value_counts()}).reindex(CLASSES)

In [ ]:
# Show some of the contradictory pictures so you can see why they were dropped.
if len(conflicts):
    groups = list(conflicts.groupby("md5"))[:6]
    fig, axes = plt.subplots(1, len(groups), figsize=(3.2 * len(groups), 3.4))
    axes = np.atleast_1d(axes)
    for a, (_, g) in zip(axes, groups):
        with Image.open(g.iloc[0].path) as im:
            a.imshow(im.convert("RGB"))
        a.set_title("\n".join(sorted(set(g.label))), fontsize=9)
        a.axis("off")
    fig.suptitle("Identical files filed under different classes (dropped)")
    plt.tight_layout()
    plt.show()

In [ ]:
if SMOKE_TEST:  # tiny subset so the whole notebook runs in minutes
    orig = clean[~clean.is_preaug].sample(frac=1, random_state=CFG["seed"]).groupby("label").head(40)
    clean = pd.concat([orig, clean[clean.is_preaug].head(20)]).reset_index(drop=True)
    print("SMOKE_TEST subset:", clean.label.value_counts().to_dict())

### 4.2 Near-duplicates

Some images are not byte-identical but are the same photo after resizing, colour changes, flips, crops or rotations (several have the smeared borders typical of Keras `ImageDataGenerator`). If one copy lands in training and another in the test set, test accuracy is inflated.

Groups of copies are found in two steps.

1. **Embedding similarity.** Each image becomes a 1,280-number feature vector from an ImageNet-pretrained EfficientNet-B0 (no labels used), also compared against its mirror image. Pairs with cosine similarity of at least `near_dup_threshold` (0.86) are linked directly. Pairs inspected at that level were almost always the same photo.
2. **Keypoint check.** Below 0.86, similarity alone cannot tell a rotated or colour-shifted copy from a different photo of a similar mouth. Each photo is therefore compared with its `geo_neighbours` most similar photos (similarity 0.60 to 0.86) using OpenCV: ORB keypoints are matched on contrast-equalised greyscale versions (and against the mirror image), and the pair is linked when at least `orb_min_inliers` matches agree on one geometric transform (RANSAC homography) covering at least 15% of both images. Rotation, cropping and colour changes keep that geometry; a different photo does not. This step runs on the CPU and takes about 1 to 3 minutes on Colab.

Measured on this dataset while building the notebook: with step 1 alone, 33 to 44% of test photos had a copy in training that a slower, more thorough version of the step 2 check could detect. With both steps it was 2 to 4%. Lowering the step 1 threshold instead does not work: below about 0.85, chains of merely similar photos merge into groups of 100 to 400 images.

Linked photos form a **group**, and section 5 keeps every group inside a single split. Different shots of the same mouth, as opposed to copies of one photo, can still be split apart because the dataset has no patient IDs, so treat test scores as slightly optimistic. Check the three panels this section prints.

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


def load_image(source, max_side=CFG["max_side"]):
    # The one loading function for training, evaluation and the app, so preprocessing cannot drift.
    # source: file path, file-like object or PIL image.
    # Fixes phone-photo rotation from EXIF and shrinks the longest side to max_side.
    img = source if isinstance(source, Image.Image) else Image.open(source)
    if img.format == "JPEG":
        img.draft("RGB", (max_side, max_side))   # fast JPEG decode at reduced scale
    img = ImageOps.exif_transpose(img).convert("RGB")
    img.thumbnail((max_side, max_side), Image.Resampling.BICUBIC)
    return img


eval_tf = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class PathDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths, self.transform = list(paths), transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.transform(load_image(self.paths[i]))


@torch.no_grad()
def embed(paths):
    m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    m.classifier = nn.Identity()
    m = m.to(device).eval()
    loader = DataLoader(PathDataset(paths, eval_tf), batch_size=64, num_workers=CFG["num_workers"])
    e, ef = [], []
    for x in loader:
        x = x.to(device)
        e.append(nn.functional.normalize(m(x), dim=1).cpu())
        ef.append(nn.functional.normalize(m(torch.flip(x, dims=[3])), dim=1).cpu())
    return torch.cat(e), torch.cat(ef)


t0 = time.time()
E, E_flip = embed(clean.path)
S = torch.maximum(E @ E.T, E @ E_flip.T)
S = torch.maximum(S, S.T)
S.fill_diagonal_(-1.0)
print(f"Embedded {len(clean)} images in {time.time() - t0:.0f}s")

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor
import cv2

cv2.setNumThreads(1)              # parallelism comes from the thread pool below
_tls = threading.local()


def orb_features(path):
    # ORB keypoints of the contrast-equalised greyscale image and of its mirror image.
    if not hasattr(_tls, "orb"):
        _tls.orb = cv2.ORB_create(nfeatures=CFG["orb_features"])
    g = cv2.equalizeHist(np.asarray(load_image(path).convert("L")))
    out = []
    for im in (g, np.ascontiguousarray(g[:, ::-1])):
        kp, des = _tls.orb.detectAndCompute(im, None)
        pts = np.float32([k.pt for k in kp]) if kp else np.zeros((0, 2), np.float32)
        out.append((pts, des, im.shape))
    return out


def geometric_match(fa, fb, min_inliers=CFG["orb_min_inliers"], min_cover=0.15):
    # True when enough keypoint matches agree on one geometric transform (RANSAC homography)
    # and the agreeing points cover at least min_cover of both images.
    (pa, da, sa), (pb, db, sb) = fa, fb
    if da is None or db is None or len(da) < min_inliers or len(db) < min_inliers:
        return False
    knn = cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(da, db, k=2)
    good = [m for m, m2 in (p for p in knn if len(p) == 2) if m.distance < 0.75 * m2.distance]  # Lowe ratio test
    if len(good) < min_inliers:
        return False
    a, b = pa[[m.queryIdx for m in good]], pb[[m.trainIdx for m in good]]
    H, mask = cv2.findHomography(a, b, cv2.RANSAC, 8.0)
    if H is None:
        return False
    inl = mask.ravel().astype(bool)
    if inl.sum() < min_inliers:
        return False
    cover = lambda pts, shape: np.ptp(pts[:, 0]) * np.ptp(pts[:, 1]) / (shape[0] * shape[1])
    return min(cover(a[inl], sa), cover(b[inl], sb)) >= min_cover


# Candidate pairs: each original photo and its most similar photos below the direct-link threshold.
orig_idx = np.where(~clean.is_preaug.values)[0]
S_orig = S[orig_idx][:, orig_idx]
k = min(CFG["geo_neighbours"], len(orig_idx) - 1)
top_sim, top_idx = torch.topk(S_orig, k, dim=1)
cand = sorted({(min(i, j), max(i, j))
               for i in range(len(orig_idx))
               for s, j in zip(top_sim[i].tolist(), top_idx[i].tolist())
               if CFG["geo_min_sim"] <= s < CFG["near_dup_threshold"]})

t0 = time.time()
with ThreadPoolExecutor(max_workers=os.cpu_count() or 2) as pool:
    feats = list(pool.map(orb_features, clean.path.values[orig_idx]))
    same = list(pool.map(lambda p: geometric_match(feats[p[0]][0], feats[p[1]][0])
                         or geometric_match(feats[p[0]][0], feats[p[1]][1]), cand))
verified = [(int(orig_idx[a]), int(orig_idx[b])) for (a, b), ok in zip(cand, same) if ok]
rejected = [(int(orig_idx[a]), int(orig_idx[b])) for (a, b), ok in zip(cand, same) if not ok]
print(f"Keypoint check: {len(cand)} candidate pairs, {len(verified)} confirmed as the same photo "
      f"({time.time() - t0:.0f}s)")

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]
            a = self.p[a]
        return a

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[ra] = rb


n = len(clean)
is_pre = clean.is_preaug.values

# 1. Link original photos by embedding similarity (direct) and by the keypoint check.
uf = UnionFind(n)
ii, jj = torch.where(torch.triu(S_orig >= CFG["near_dup_threshold"], diagonal=1))
emb_links = [(int(orig_idx[a]), int(orig_idx[b])) for a, b in zip(ii.tolist(), jj.tolist())]
for a, b in emb_links + verified:
    uf.union(a, b)
group = np.array([uf.find(i) for i in range(n)])

# 2. Each pre-augmented discoloration file joins the group of its most similar original
#    discoloration photo. Pre-augmented files are not linked to each other, so they
#    cannot chain unrelated originals together.
pre_idx = np.where(is_pre)[0]
orig_td_idx = np.where((clean.label.values == "tooth discoloration") & ~is_pre)[0]
if len(pre_idx) and len(orig_td_idx):
    best_sim, best = S[pre_idx][:, orig_td_idx].max(dim=1)
    group[pre_idx] = group[orig_td_idx[best.numpy()]]
    print(f"Pre-augmented files matched to an original with median similarity {best_sim.median().item():.2f} "
          f"(approximate, see 4.3)")

clean["group"] = group
clean["pos"] = np.arange(n)            # row position in S, used by the leakage check in section 5
gs = clean.loc[~is_pre, "group"].value_counts()
mixed = clean.groupby("group").label.nunique()
print(f"Links between original photos: {len(emb_links)} by embedding similarity "
      f">= {CFG['near_dup_threshold']}, {len(verified)} by the keypoint check")
print(f"Original photos: {int(gs.sum())} in {len(gs)} groups | photos sharing a group: {int(gs[gs > 1].sum())} "
      f"| largest group: {int(gs.max())}")
print(f"Groups containing more than one class: {int((mixed > 1).sum())} (possible label noise, kept)")

In [ ]:
def show_pairs(pairs, title):
    if not pairs:
        print("No pairs in this band.")
        return
    fig, axes = plt.subplots(2, len(pairs), figsize=(2.4 * len(pairs), 5))
    axes = np.array(axes).reshape(2, -1)
    for k, (a, b, s) in enumerate(pairs):
        for r, idx in enumerate((a, b)):
            with Image.open(clean.path[idx]) as im:
                axes[r, k].imshow(ImageOps.exif_transpose(im).convert("RGB"))
            axes[r, k].set_title(f"{clean.label[idx][:12]}" + (f"\nsim {s:.2f}" if r == 0 else ""), fontsize=8)
            axes[r, k].axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def sample_pairs(pairs, k=8):
    pairs = list(pairs)
    if not pairs:
        return []
    pick = np.random.default_rng(CFG["seed"]).choice(len(pairs), size=min(k, len(pairs)), replace=False)
    return [(a, b, float(S[a, b])) for a, b in (pairs[i] for i in pick)]


th = CFG["near_dup_threshold"]
show_pairs(sample_pairs([p for p in emb_links if S[p] < th + 0.04]),
           f"Linked by embedding similarity ({th:.2f} to {th + 0.04:.2f}). Top and bottom should be the same photo.")
show_pairs(sample_pairs(verified), "Linked by the keypoint check (similarity below threshold). Should be the same photo.")
show_pairs(sample_pairs([p for p in rejected if S[p] >= th - 0.06]),
           f"Rejected by the keypoint check (similarity {th - 0.06:.2f} to {th:.2f}). Mostly different photos is good.")

### 4.3 Pre-augmented files

875 of the 1,058 tooth discoloration files are named `Tooth_Discoloration_0_<n>.jpeg`, the naming pattern of Keras `ImageDataGenerator` output. They are augmented copies of the remaining originals, many with strong colour shifts. For a class defined by tooth colour, colour-shifted copies can teach the wrong cue, and they inflate the class count.

- `use_preaugmented=False` (default): drop them. Training uses its own on-the-fly augmentation instead, without hue changes.
- `use_preaugmented=True`: keep them in **training only**, and only when their matched source image (found in 4.2) is also in training. The match is approximate. The median similarity to the best-matching original is only about 0.74, so some copies may be matched to the wrong source and leak information about validation or test photos. That is why the default drops them.

Either way, validation and test contain only original photos.

## 5. Train / validation / test split

About 72% of original photos go to training, 14% to validation (model selection, early stopping, confidence threshold) and 14% to test (used once, in section 7).

- Groups with more than `max_eval_group` (30) photos always go to training. They are dozens of variants of a few photos, and in the test set they would count those photos dozens of times.
- `StratifiedGroupKFold` cuts the remaining photos into 20 folds with similar class proportions, never splitting a group. Three randomly chosen folds become the test set and three the validation set. The random choice matters: `StratifiedGroupKFold` places the largest groups in the first folds, so using fold 0 as the test set would fill it with one cluster of near-identical photos.

In [ ]:
def assign_splits(frame, seed, n_folds=20):
    split = np.full(len(frame), "train", dtype=object)
    sizes = frame.group.map(frame.group.value_counts()).values
    rest = np.where(sizes <= CFG["max_eval_group"])[0]          # big clusters stay in training
    sub = frame.iloc[rest]
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    fold = np.empty(len(rest), dtype=int)
    for k, (_, idx) in enumerate(sgkf.split(sub, sub.label_idx, sub.group)):
        fold[idx] = k
    order = np.random.default_rng(seed).permutation(n_folds)   # random folds, not the biased fold 0
    n_test, n_val = round(CFG["test_frac"] * n_folds), round(CFG["val_frac"] * n_folds)
    split[rest[np.isin(fold, order[:n_test])]] = "test"
    split[rest[np.isin(fold, order[n_test:n_test + n_val])]] = "val"
    return split


originals = clean[~clean.is_preaug].copy()
originals["split"] = assign_splits(originals, CFG["seed"])
big = originals.group.map(originals.group.value_counts()) > CFG["max_eval_group"]
print(f"{int(big.sum())} photos in groups larger than {CFG['max_eval_group']} kept in training")
train_df, val_df, test_df = (originals[originals.split == s] for s in ("train", "val", "test"))

if CFG["use_preaugmented"]:
    extra = clean[clean.is_preaug & clean.group.isin(set(train_df.group))]
    train_df = pd.concat([train_df, extra])
    print(f"Added {len(extra)} pre-augmented files to training; "
          f"{clean.is_preaug.sum() - len(extra)} dropped because their source is in val/test")
else:
    print(f"Dropped {clean.is_preaug.sum()} pre-augmented files")

train_df, val_df, test_df = (d.reset_index(drop=True) for d in (train_df, val_df, test_df))

# Leakage checks: no group and no identical file may appear in two splits.
for a_name, a in [("train", train_df), ("val", val_df)]:
    for b_name, b in [("val", val_df), ("test", test_df)]:
        if a_name == b_name:
            continue
        assert not set(a.group) & set(b.group), f"group overlap {a_name}/{b_name}"
        assert not set(a.md5) & set(b.md5), f"file overlap {a_name}/{b_name}"
print("Leakage checks passed: no shared groups or files between splits.")

# Residual leakage: how close does each test photo get to the training set? (embedding similarity from 4.2)
nn_train = S[torch.tensor(test_df.pos.values)][:, torch.tensor(train_df.pos.values)].max(dim=1).values.numpy()
print(f"Test photos whose most similar training image has similarity >= 0.85: {(nn_train >= 0.85).mean():.1%} "
      f"(>= 0.80: {(nn_train >= 0.80).mean():.1%})")
largest_share = max(g.group.value_counts().iloc[0] / len(g) for _, g in test_df.groupby("label"))
print(f"Largest single group's share of one class's test photos: {largest_share:.0%}")

split_table = pd.DataFrame({"train": train_df.label.value_counts(), "val": val_df.label.value_counts(),
                            "test": test_df.label.value_counts()}).reindex(CLASSES).fillna(0).astype(int)
split_table.loc["TOTAL"] = split_table.sum()
pd.concat([train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test")])[
    ["split", "label", "path", "md5", "group"]].to_csv(ART_DIR / "splits.csv", index=False)
split_table

## 6. Training

### 6.1 Datasets, augmentation and loss

Training augmentation uses random crops, flips, small rotations, brightness/contrast changes and occasional blur (to weaken any sharpness shortcut). It deliberately avoids hue and saturation changes, because colour is the evidence for discoloration and calculus.

The loss is cross-entropy with class weights `N / (K * n_class)` from the training split and label smoothing of 0.1.

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(CFG["img_size"], scale=(0.6, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomApply([transforms.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class OralDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame.path.tolist()
        self.labels = frame.label_idx.tolist()
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.transform(load_image(self.paths[i])), self.labels[i]


def make_loader(frame, transform, shuffle):
    return DataLoader(OralDataset(frame, transform), batch_size=CFG["batch_size"], shuffle=shuffle,
                      drop_last=shuffle, num_workers=CFG["num_workers"], pin_memory=device.type == "cuda",
                      persistent_workers=CFG["num_workers"] > 0)


train_loader = make_loader(train_df, train_tf, shuffle=True)
val_loader = make_loader(val_df, eval_tf, shuffle=False)
test_loader = make_loader(test_df, eval_tf, shuffle=False)

n_c = train_df.label_idx.value_counts().reindex(range(len(CLASSES))).fillna(0).values
class_weights = torch.tensor(len(train_df) / (len(CLASSES) * np.maximum(n_c, 1)), dtype=torch.float32)
print("Class weights:", dict(zip(CLASSES, class_weights.numpy().round(2))))

# Sanity check: one augmented batch (de-normalised for display)
xb, yb = next(iter(train_loader))
grid = (xb[:8].permute(0, 2, 3, 1).numpy() * STD + MEAN).clip(0, 1)
fig, axes = plt.subplots(1, 8, figsize=(20, 3))
for a, im, y in zip(axes, grid, yb[:8]):
    a.imshow(im)
    a.set_title(CLASSES[y], fontsize=9)
    a.axis("off")
fig.suptitle("Augmented training batch")
plt.show()

### 6.2 Models and training loop

`build_model` returns a pretrained backbone with a new 6-class head.

Each run trains the head alone for `warmup_epochs` with the backbone frozen (BatchNorm layers kept in eval mode, so their statistics do not drift). Then it unfreezes the backbone with a 10x lower learning rate. AdamW with cosine learning-rate decay, mixed precision on the GPU, and gradient clipping are used throughout. The checkpoint with the best **validation macro-F1** is kept, and training stops after `patience` epochs without improvement.

In [ ]:
def build_model(arch, num_classes, dropout=0.3, pretrained=True):
    # pretrained=False is for loading a saved checkpoint: skips the ImageNet download.
    w = lambda weights: weights.DEFAULT if pretrained else None
    if arch == "mobilenet_v3_small":
        m = models.mobilenet_v3_small(weights=w(models.MobileNet_V3_Small_Weights))
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
    elif arch == "mobilenet_v3_large":
        m = models.mobilenet_v3_large(weights=w(models.MobileNet_V3_Large_Weights))
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
    elif arch == "efficientnet_b0":
        m = models.efficientnet_b0(weights=w(models.EfficientNet_B0_Weights))
        m.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(m.classifier[1].in_features, num_classes))
    elif arch == "convnext_tiny":
        m = models.convnext_tiny(weights=w(models.ConvNeXt_Tiny_Weights))
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, num_classes)
    else:
        raise ValueError(f"Unknown arch {arch}")
    # All four torchvision models share the layout features -> avgpool -> classifier.
    return m


def set_backbone_trainable(model, trainable):
    for p in model.features.parameters():
        p.requires_grad = trainable


@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x.to(device, non_blocking=True))
        ps.append(torch.softmax(logits.float(), dim=1).cpu())
        ys.append(y)
    return torch.cat(ys).numpy(), torch.cat(ps).numpy()


def run_experiment(name, arch, lr=1e-3, backbone_lr_mult=0.1, weight_decay=1e-4, warmup_epochs=2,
                   freeze_backbone=False, label_smoothing=0.1, dropout=0.3):
    seed_everything(CFG["seed"])
    model = build_model(arch, len(CLASSES), dropout).to(device)
    head_params = [p for n_, p in model.named_parameters() if not n_.startswith("features.")]
    optimizer = torch.optim.AdamW([
        {"params": model.features.parameters(), "lr": lr * backbone_lr_mult},
        {"params": head_params, "lr": lr},
    ], weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["max_epochs"])
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=label_smoothing)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    history, best_f1, best_epoch, bad_epochs = [], -1.0, 0, 0
    ckpt_path = CKPT_DIR / f"{name}.pt"
    t_start = time.time()
    for epoch in range(1, CFG["max_epochs"] + 1):
        frozen = freeze_backbone or epoch <= warmup_epochs
        set_backbone_trainable(model, not frozen)
        model.train()
        if frozen:
            model.features.eval()          # keep pretrained BatchNorm statistics fixed

        run_loss, run_correct, seen = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(optimizer)
            scaler.update()
            run_loss += loss.item() * len(y)
            run_correct += (logits.argmax(1) == y).sum().item()
            seen += len(y)
        scheduler.step()

        y_val, p_val = predict_probs(model, val_loader)
        # Same weighted, smoothed loss as training (log-probabilities are valid logits), so curves are comparable.
        val_loss = criterion(torch.log(torch.tensor(p_val) + 1e-8).to(device), torch.tensor(y_val).to(device)).item()
        pred_val = p_val.argmax(1)
        row = dict(epoch=epoch, frozen=frozen, train_loss=run_loss / seen, train_acc=run_correct / seen,
                   val_loss=val_loss, val_acc=accuracy_score(y_val, pred_val),
                   val_macro_f1=f1_score(y_val, pred_val, average="macro", zero_division=0),
                   val_bal_acc=balanced_accuracy_score(y_val, pred_val))
        history.append(row)
        improved = row["val_macro_f1"] > best_f1
        if improved:
            best_f1, best_epoch, bad_epochs = row["val_macro_f1"], epoch, 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            bad_epochs += 1
        print(f"[{name}] ep {epoch:2d} {'(frozen)' if frozen else '        '} "
              f"train loss {row['train_loss']:.3f} acc {row['train_acc']:.3f} | "
              f"val loss {val_loss:.3f} acc {row['val_acc']:.3f} macro-F1 {row['val_macro_f1']:.3f}"
              f"{'  *' if improved else ''}")
        if bad_epochs >= CFG["patience"]:
            print(f"[{name}] early stop: no val macro-F1 gain for {CFG['patience']} epochs")
            break

    best = history[best_epoch - 1]
    return dict(name=name, arch=arch, lr=lr, freeze_backbone=freeze_backbone, best_epoch=best_epoch,
                epochs_run=len(history), val_acc=best["val_acc"], val_macro_f1=best["val_macro_f1"],
                val_bal_acc=best["val_bal_acc"], params_M=sum(p.numel() for p in model.parameters()) / 1e6,
                minutes=(time.time() - t_start) / 60, ckpt=str(ckpt_path), history=history)

### 6.3 Experiments

Experiment A reproduces v1's setup (MobileNetV3-Small, frozen backbone, learning rate 1e-3) on the cleaned, leakage-free split, so you can see how much of v1's 92% came from leakage. B to D add fine-tuning and larger backbones.

On a T4, expect about 2 to 6 minutes per experiment. Add rows to `EXPERIMENTS` to try other settings (learning rate, dropout, `warmup_epochs`, `label_smoothing`). Every run is recorded in `artifacts/experiments.csv` as optimisation evidence for the rubric.

In [ ]:
EXPERIMENTS = [
    dict(name="A_mnv3s_frozen_v1_style", arch="mobilenet_v3_small", freeze_backbone=True, lr=1e-3),
    dict(name="B_mnv3s_finetune", arch="mobilenet_v3_small", lr=1e-3),
    dict(name="C_effb0_finetune", arch="efficientnet_b0", lr=1e-3),
    dict(name="D_convnext_tiny_finetune", arch="convnext_tiny", lr=5e-4, weight_decay=0.05),
]
if SMOKE_TEST:
    EXPERIMENTS = EXPERIMENTS[:2]

results = []
for exp in EXPERIMENTS:
    results.append(run_experiment(**exp))
    if device.type == "cuda":
        torch.cuda.empty_cache()

results_df = pd.DataFrame([{k: v for k, v in r.items() if k != "history"} for r in results])
results_df.to_csv(ART_DIR / "experiments.csv", index=False)
pd.DataFrame([dict(experiment=r["name"], **h) for r in results for h in r["history"]]).to_csv(
    ART_DIR / "training_history.csv", index=False)
results_df.sort_values("val_macro_f1", ascending=False).round(4)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
for r in results:
    h = pd.DataFrame(r["history"])
    ax[0].plot(h.epoch, h.train_loss, "--", label=f"{r['name']} train")
    ax[0].plot(h.epoch, h.val_loss, label=f"{r['name']} val")
    ax[1].plot(h.epoch, h.val_acc, label=r["name"])
    ax[2].plot(h.epoch, h.val_macro_f1, label=r["name"])
for a, t in zip(ax, ["Loss (dashed = train)", "Validation accuracy", "Validation macro-F1"]):
    a.set_title(t)
    a.set_xlabel("epoch")
    a.grid(alpha=0.3)
ax[2].legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "training_curves.png", dpi=120)
plt.show()

## 7. Final evaluation on the test set

The experiment with the highest validation macro-F1 is selected **before** the test set is touched. The test set is then evaluated once. Do not go back and tune on these numbers. If you change the model afterwards, report the new test result as a second attempt.

In [ ]:
best_run = max(results, key=lambda r: r["val_macro_f1"])
print(f"Selected: {best_run['name']} ({best_run['arch']}), best epoch {best_run['best_epoch']}, "
      f"val macro-F1 {best_run['val_macro_f1']:.3f}")

model = build_model(best_run["arch"], len(CLASSES)).to(device)
model.load_state_dict(torch.load(best_run["ckpt"], map_location=device, weights_only=True))
model.eval()

y_val, p_val = predict_probs(model, val_loader)
y_test, p_test = predict_probs(model, test_loader)
pred_test = p_test.argmax(1)

present = sorted(set(y_test))
try:
    auc = roc_auc_score(y_test, p_test, multi_class="ovr", average="macro", labels=list(range(len(CLASSES))))
except ValueError:
    auc = float("nan")
test_metrics = dict(accuracy=accuracy_score(y_test, pred_test),
                    balanced_accuracy=balanced_accuracy_score(y_test, pred_test),
                    macro_f1=f1_score(y_test, pred_test, average="macro", zero_division=0),
                    macro_roc_auc_ovr=auc, n_test=int(len(y_test)))
print({k: round(float(v), 4) for k, v in test_metrics.items()})
print()
print(classification_report(y_test, pred_test, labels=list(range(len(CLASSES))), target_names=CLASSES,
                            digits=3, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, pred_test, labels=list(range(len(CLASSES))))
cm_norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(1, 2, figsize=(16, 6.5))
for a, mat, title, fmt in [(ax[0], cm, "Counts", "d"), (ax[1], cm_norm, "Row-normalised (recall on diagonal)", ".2f")]:
    a.imshow(mat, cmap="Blues")
    a.set_xticks(range(len(CLASSES)), CLASSES, rotation=35, ha="right")
    a.set_yticks(range(len(CLASSES)), CLASSES)
    a.set_xlabel("Predicted")
    a.set_ylabel("True")
    a.set_title(f"Test confusion matrix: {title}")
    for i in range(len(CLASSES)):
        for j in range(len(CLASSES)):
            a.text(j, i, format(mat[i, j], fmt), ha="center", va="center",
                   color="white" if mat[i, j] > mat.max() / 2 else "black")
plt.tight_layout()
plt.savefig(FIG_DIR / "confusion_matrix.png", dpi=120)
plt.show()

### 7.1 How to read these results

- **Recall** for a class is the share of that class's test images the model found. For a screening aid, low recall on caries or Mouth ulcer means missed conditions. Treat that as the most important failure.
- **Precision** is how often a prediction of that class was right. Low precision means false alarms.
- **Macro-F1 and balanced accuracy** weight every class equally, so the small classes count as much as Calculus and Hypodontia.
- Compare experiment A's validation scores here with v1's 92%. The gap shows how much v1's number depended on duplicates leaking between splits.
- The test set is small for some classes (roughly 25 to 45 images). One or two images change a class's recall by several percentage points, so avoid over-interpreting small differences.

### 7.2 Misclassified test images

In [ ]:
wrong = np.where(pred_test != y_test)[0]
print(f"{len(wrong)} of {len(y_test)} test images misclassified")
show = wrong[:12]
if len(show):
    cols = 6
    rows = int(np.ceil(len(show) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.3))
    axes = np.atleast_1d(axes).ravel()
    for a in axes:
        a.axis("off")
    for a, i in zip(axes, show):
        a.imshow(load_image(test_df.path[i]))
        a.set_title(f"true: {CLASSES[y_test[i]]}\npred: {CLASSES[pred_test[i]]} ({p_test[i].max():.2f})", fontsize=8)
    plt.tight_layout()
    plt.show()

### 7.3 Shortcut check: results by source resolution

If the model had learnt "large phone photo = Normal", non-Normal images from phones would be predicted Normal and Normal images that came from the web would be missed. This table shows test accuracy and the share of Normal predictions by the image's **original** resolution.

In [ ]:
res = test_df.assign(pred=[CLASSES[k] for k in pred_test], correct=pred_test == y_test,
                     source=pd.cut(test_df.long_side, [0, 700, 2000, 1e9], labels=["<=700px", "701-2000px", ">2000px"]))
shortcut = res.groupby(["label", "source"], observed=True).agg(
    n=("correct", "size"), accuracy=("correct", "mean"), predicted_normal=("pred", lambda s: (s == "Normal").mean()))
shortcut.round(2)

Look for rows where the class is not Normal, the source is large, and `predicted_normal` is high. That pattern points to a resolution or camera shortcut. Many cells will be empty because the dataset has few phone photos outside Normal. That gap is itself an omission-bias finding worth noting in the Ethical AI checklist (Q5 and Q9).

## 8. Confidence threshold and "uncertain" fallback

A threshold is chosen on the **validation** set. Predictions below it are reported as "uncertain, please see a dentist" and are not given as a diagnosis. The threshold is the lowest confidence at which validation accuracy on the predictions kept reaches `target_selective_acc`. The test set then shows how this holds up on unseen data.

In [ ]:
def selective_curve(y, p, thresholds):
    conf, pred = p.max(1), p.argmax(1)
    rows = []
    for t in thresholds:
        keep = conf >= t
        rows.append(dict(threshold=t, coverage=keep.mean(),
                         selective_acc=(pred[keep] == y[keep]).mean() if keep.any() else np.nan))
    return pd.DataFrame(rows)


thresholds = np.round(np.arange(0.30, 0.991, 0.01), 2)
val_curve = selective_curve(y_val, p_val, thresholds)
MIN_KEPT = min(20, len(y_val) // 2)   # ignore thresholds that keep too few images to judge accuracy
ok = val_curve[(val_curve.selective_acc >= CFG["target_selective_acc"]) &
               (val_curve.coverage * len(y_val) >= MIN_KEPT)]
if len(ok):
    CONF_THRESHOLD = float(ok.threshold.min())
else:
    CONF_THRESHOLD = 0.90
    print(f"Validation accuracy never reached {CFG['target_selective_acc']:.0%} on at least {MIN_KEPT} images; "
          f"falling back to 0.90. Consider a lower target_selective_acc or a better model.")

test_curve = selective_curve(y_test, p_test, [CONF_THRESHOLD])
print(f"Chosen threshold: {CONF_THRESHOLD:.2f}")
print(f"Test: answers {test_curve.coverage[0]:.1%} of images, accuracy on those {test_curve.selective_acc[0]:.1%}, "
      f"{(1 - test_curve.coverage[0]):.1%} flagged uncertain")

fig, a = plt.subplots(figsize=(7, 4))
a.plot(val_curve.threshold, val_curve.coverage, label="coverage (val)")
a.plot(val_curve.threshold, val_curve.selective_acc, label="accuracy on kept (val)")
a.axvline(CONF_THRESHOLD, color="red", ls="--", label=f"chosen {CONF_THRESHOLD:.2f}")
a.set_xlabel("confidence threshold")
a.legend()
a.grid(alpha=0.3)
plt.savefig(FIG_DIR / "confidence_threshold.png", dpi=120)
plt.show()

## 9. Grad-CAM focus circle: where the model looked

Grad-CAM scores how much each image region raised the predicted class. Instead of showing the raw colour heatmap, we threshold it and draw a circle around the strongest region on the original photo, so a user can see the area at a glance. The circle marks what the model looked at. It does **not** locate the lesion and must not be presented as a diagnosis. Check whether the circle falls on teeth and gums, or on background, lips, lighting or borders (signs of a shortcut).

In [ ]:
import cv2
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

DISPLAY_SIZE = 512          # draw on a larger image so the circle lines stay sharp


def gradcam_heat(model, pil_img, class_idx):
    # Returns the Grad-CAM map (img_size x img_size, values 0..1) for one class.
    for p in model.parameters():        # CAM needs gradients through the backbone
        p.requires_grad_(True)
    model.eval()
    x = eval_tf(pil_img).unsqueeze(0).to(device)
    with GradCAM(model=model, target_layers=[model.features[-1]]) as cam:
        return cam(input_tensor=x, targets=[ClassifierOutputTarget(int(class_idx))])[0]


def focus_regions(heat, rel_thresh=0.6, max_regions=1, min_area=0.005):
    # Keep pixels above rel_thresh * peak, split them into connected blobs and
    # return the strongest blobs as (peak, x, y, w, h), strongest first.
    mask = (heat >= rel_thresh * heat.max()).astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    regions = []
    for i in range(1, n):
        x, y, w, h, area = stats[i]
        if area < min_area * mask.size:
            continue
        regions.append((float(heat[labels == i].max()), int(x), int(y), int(w), int(h)))
    return sorted(regions, reverse=True)[:max_regions]


def draw_focus(model, pil_img, class_idx, max_regions=1, size=DISPLAY_SIZE):
    # Photo with a circle around the region(s) that most influenced class_idx.
    heat = cv2.resize(gradcam_heat(model, pil_img, class_idx), (size, size))
    out = np.ascontiguousarray(np.asarray(pil_img.convert("RGB").resize((size, size))), dtype=np.uint8).copy()
    thick = max(2, size // 128)
    for rank, (_, x, y, w, h) in enumerate(focus_regions(heat, max_regions=max_regions)):
        centre = (x + w // 2, y + h // 2)
        radii = (max(w // 2 + size // 25, size // 16), max(h // 2 + size // 25, size // 16))
        colour = (0, 255, 0) if rank == 0 else (255, 220, 0)   # green = strongest, yellow = next
        cv2.ellipse(out, centre, radii, 0, 0, 360, (0, 0, 0), thick + 3, cv2.LINE_AA)  # dark outline for contrast
        cv2.ellipse(out, centre, radii, 0, 0, 360, colour, thick, cv2.LINE_AA)
    return out


fig, axes = plt.subplots(2, len(CLASSES), figsize=(3.2 * len(CLASSES), 7))
for k, c in enumerate(CLASSES):
    idx = np.where((y_test == k) & (pred_test == k))[0]
    for a in axes[:, k]:
        a.axis("off")
    if len(idx) == 0:
        axes[0, k].set_title(f"{c}\n(no correct test example)", fontsize=9)
        continue
    i = idx[0]
    img = load_image(test_df.path[i])
    axes[0, k].imshow(img.resize((CFG["img_size"], CFG["img_size"])))
    axes[0, k].set_title(f"{c}\nconf {p_test[i].max():.2f}", fontsize=9)
    axes[1, k].imshow(draw_focus(model, img, k))
    axes[1, k].set_title("area the model focused on", fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "gradcam_examples.png", dpi=120)
plt.show()

## 10. Save the model for the app

`artifacts/dental_classifier.pt` holds the weights plus everything the app needs to reproduce preprocessing: class names, input size, `max_side`, normalisation and the confidence threshold. To use the model in your app (FastAPI, Streamlit, Gradio, etc.), copy `build_model`, `load_image`, `load_classifier` and `predict_image`. `load_classifier` builds the network without downloading ImageNet weights, because the checkpoint replaces them.

In [ ]:
bundle = dict(
    arch=best_run["arch"], state_dict=model.state_dict(), classes=CLASSES, img_size=CFG["img_size"],
    max_side=CFG["max_side"], mean=MEAN, std=STD, conf_threshold=CONF_THRESHOLD,
    test_metrics={k: float(v) for k, v in test_metrics.items()}, experiment=best_run["name"],
    split_seed=CFG["seed"], trained_at=time.strftime("%Y-%m-%d %H:%M:%S"),
)
MODEL_PATH = ART_DIR / "dental_classifier.pt"
torch.save(bundle, MODEL_PATH)
meta = {k: v for k, v in bundle.items() if k != "state_dict"}
(ART_DIR / "model_card.json").write_text(json.dumps(meta, indent=2))
print("Saved", MODEL_PATH, f"({MODEL_PATH.stat().st_size / 1e6:.1f} MB)")
print(json.dumps(meta, indent=2))

In [ ]:
def load_classifier(path, map_location="cpu"):
    b = torch.load(path, map_location=map_location, weights_only=True)
    m = build_model(b["arch"], len(b["classes"]), pretrained=False)
    m.load_state_dict(b["state_dict"])
    return m.eval(), b


def predict_image(model, bundle, image):
    # image: file path, file-like object or PIL image. Returns a dict for the UI.
    img = load_image(image, bundle["max_side"])          # same preprocessing as training
    tf = transforms.Compose([transforms.Resize((bundle["img_size"], bundle["img_size"])),
                             transforms.ToTensor(), transforms.Normalize(bundle["mean"], bundle["std"])])
    dev = next(model.parameters()).device
    with torch.no_grad():
        probs = torch.softmax(model(tf(img).unsqueeze(0).to(dev)), dim=1)[0].cpu().numpy()
    order = probs.argsort()[::-1]
    top = order[0]
    uncertain = probs[top] < bundle["conf_threshold"]
    return dict(
        label=None if uncertain else bundle["classes"][top],
        confidence=float(probs[top]),
        uncertain=bool(uncertain),
        top3=[(bundle["classes"][i], float(probs[i])) for i in order[:3]],
        message=("The image is unclear to the model. Please consult a dentist." if uncertain else
                 "AI screening result only, not a diagnosis. Please consult a dentist."),
    )


# Round-trip check: reload from disk and compare with the in-memory model on one test image.
clf, clf_bundle = load_classifier(MODEL_PATH, map_location=device)
check = predict_image(clf, clf_bundle, test_df.path[0])
print(check)
assert abs(check["confidence"] - float(p_test[0].max())) < 1e-2, "reloaded model disagrees with evaluation"
print("Reload check passed.")

## 11. Try your own photo

In [ ]:
if IN_COLAB:
    from google.colab import files
    print("Upload a dental photo:")
    uploaded = files.upload()
    paths = list(uploaded.keys())
else:
    paths = [test_df.path[0]]   # local run: use a test image

for pth in paths:
    out = predict_image(clf, clf_bundle, pth)
    img = load_image(pth)
    print(f"\n{Path(pth).name}")
    print("Result:", "UNCERTAIN" if out["uncertain"] else out["label"], f"(confidence {out['confidence']:.1%})")
    print("Top 3:", ", ".join(f"{c} {p:.1%}" for c, p in out["top3"]))
    print(out["message"])

    show_cam = not out["uncertain"] and out["label"] != "Normal"
    fig, ax = plt.subplots(1, 2 if show_cam else 1, figsize=(10 if show_cam else 5, 5))
    ax = np.atleast_1d(ax)
    ax[0].imshow(img.resize((CFG["img_size"], CFG["img_size"])))
    ax[0].set_title("Uploaded image")
    if show_cam:
        ax[1].imshow(draw_focus(clf, img, CLASSES.index(out["label"])))
        ax[1].set_title(f"Area the model focused on for '{out['label']}'")
        fig.text(0.5, 0.02, "Green circle = area that most influenced the result. It is not a lesion outline.",
                 ha="center", fontsize=9)
    for a in ax:
        a.axis("off")
    plt.show()

## 12. Download or keep the results

Colab deletes `/content` when the session ends. Run one of the cells below to keep `artifacts/` (model, model card, experiment table, training history, split file and figures).

In [ ]:
# Option A: download a zip to your computer
if IN_COLAB:
    zip_path = shutil.make_archive("/content/artifacts", "zip", ART_DIR)
    from google.colab import files
    files.download(zip_path)

In [ ]:
# Option B: copy to Google Drive (set to True to use)
SAVE_TO_DRIVE = False
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/IT2226_artifacts") / time.strftime("%Y%m%d_%H%M")
    shutil.copytree(ART_DIR, dest)
    print("Copied to", dest)

## 13. Notes for your report, AI Use Log and Ethical AI checklist

- **AI Use Log.** This notebook was generated with an AI assistant. Commit it as its own bulk import before your edits, and log what you verified: the leakage asserts in section 5, the threshold panels in 4.2, and the shortcut table in 7.3.
- **Checklist Q5 / Q9 (data relevance and bias).** Report the class imbalance, the resolution difference between Normal and the other classes, the 27 pictures with conflicting labels, and that the data are web and phone photos, not clinical intra-oral camera images of Singapore patients.
- **Checklist Q10 (distinct splits).** Point to `artifacts/splits.csv`, the group-aware split, the keypoint check in 4.2 and the leakage asserts in section 5. The test set was evaluated once, after model selection on validation.
- **Checklist Q2 (human involvement).** The app gives a screening suggestion, flags low-confidence images as uncertain, and always tells users to consult a dentist.
- **Checklist Q11 (explainability).** Use the Grad-CAM figures, with the caveat that they show what influenced the model, not where the disease is.
- **Limitations.** Small test counts for minority classes; possible label noise (mixed-class groups in 4.2); different shots of the same mouth can still cross splits because the dataset has no patient IDs (section 5 prints how close test photos get to training photos); web images may not match the photos real users will take.